# Dataset 4 Pipeline

This notebook holds the staged workflow and code cells for dataset `4`.
The portal routes here as soon as you choose dataset `4`.
The cells below keep one code cell per workflow headline for the compact gene-index and chemistry branch.


### Dataset 4 Example Row

<div style="margin:0.5rem 0 0.9rem 0;">
<div style="font-size:11px; margin-bottom:0.5rem; line-height:1.35;">
<span style="color:#79c0ff; font-weight:600;">blue = comp_name</span>
<span style="color:#c297ff; font-weight:600; margin-left:12px;">violet = gene_idx block (3 values)</span>
<span style="color:#f2cc60; font-weight:600; margin-left:12px;">gold = compact uint8 fingerprint block (96 values)</span>
<span style="color:#ff7b72; font-weight:600; margin-left:12px;">red = labels</span>
</div>
<div style="background:#0d1117; border:1px solid #30363d; border-radius:10px; padding:10px 12px; max-height:480px; overflow:auto;">
<pre style="margin:0; white-space:pre-wrap; word-break:break-word; line-height:1.34; color:#f8fafc; font-size:10px;"><span style="color:#79c0ff;">ENSG00000000003_amide,</span>
<span style="color:#c297ff;">0,0,3,</span>
<span style="color:#f2cc60;">76.0,196.0,0.0,0.0,0.0,4.0,16.0,0.0,10.0,16.0,132.0,32.0,32.0,0.0,148.0,16.0,128.0,168.0,216.0,128.0,0.0,0.0,3.0,65.0,</span>
<span style="color:#f2cc60;">0.0,64.0,0.0,128.0,16.0,97.0,0.0,0.0,8.0,2.0,0.0,129.0,1.0,128.0,8.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,</span>
<span style="color:#f2cc60;">96.0,2.0,0.0,2.0,1.0,0.0,0.0,128.0,2.0,65.0,2.0,66.0,80.0,0.0,1.0,88.0,0.0,0.0,0.0,0.0,1.0,4.0,0.0,32.0,</span>
<span style="color:#f2cc60;">64.0,4.0,192.0,32.0,40.0,0.0,128.0,0.0,64.0,32.0,128.0,192.0,1.0,0.0,1.0,1.0,0.0,25.0,66.0,0.0,64.0,64.0,2.0,16.0,</span>
<span style="color:#ff7b72;">-0.1571126435579587,0</span></pre>
</div>
</div>


In [ ]:
# Dataset 4.1 - Rebuild normalized_rawcounts.csv from 6048D_rawCounts.txt.
# Details:
# - Reads the raw-count file from this dataset root.
# - Recomputes normalized_rawcounts.csv from scratch.
# - Writes the generated table to working/normalized_rawcounts.csv.

from pathlib import Path
import pandas as pd

DATA_ROOT = Path("/media/volume/sirna-features")
DATASET_DIR = DATA_ROOT / "dataset4"
WORKING_DIR = DATASET_DIR / "working"
WORKING_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS_OF_INTEREST = [
    "MR10_unmod_1",
    "MR11_unmod_2",
    "MR12_unmod_3",
    "MR1_NT_1",
    "MR2_NT_2",
    "MR3_NT_3",
    "MR4_Amide3_1",
    "MR5_Amide3_2",
    "MR6_Amide3_3",
    "MR7_GNA7_1",
    "MR8_GNA7_2",
    "MR9_GNA7_3",
]
NT_COLS = ["MR1_NT_1", "MR2_NT_2", "MR3_NT_3"]
UNMOD_COLS = ["MR10_unmod_1", "MR11_unmod_2", "MR12_unmod_3"]
AMIDE_COLS = ["MR4_Amide3_1", "MR5_Amide3_2", "MR6_Amide3_3"]
GNA_COLS = ["MR7_GNA7_1", "MR8_GNA7_2", "MR9_GNA7_3"]

raw_counts = pd.read_csv(DATASET_DIR / "6048D_rawCounts.txt", sep="\t", index_col=0)
counts_only = raw_counts[COLUMNS_OF_INTEREST].copy()

total_counts = counts_only.sum(axis=0)
scaling_factors = total_counts / total_counts.mean()
normalized = counts_only.div(scaling_factors, axis=1)

normalized["mean_nt"] = normalized[NT_COLS].mean(axis=1)
normalized["mean_unmod"] = normalized[UNMOD_COLS].mean(axis=1)
normalized["mean_amide"] = normalized[AMIDE_COLS].mean(axis=1)
normalized["mean_gna"] = normalized[GNA_COLS].mean(axis=1)
normalized.insert(0, "ensembl_id", normalized.index)

normalized.to_csv(WORKING_DIR / "normalized_rawcounts.csv", index=False)
print(normalized.shape)
normalized.head(3)


In [ ]:
# Dataset 4.2 - Rebuild the alignment labels and save one working sequence example.
# Details:
# - Reads gene_alignments3.csv from this dataset root.
# - Recomputes log2FC and off_target in memory from normalized counts.
# - Retrieves one full transcript example and writes sequence_example.json/.fasta into working.

from pathlib import Path
import json
import numpy as np
import pandas as pd
import requests

DATA_ROOT = Path("/media/volume/sirna-features")
DATASET_DIR = DATA_ROOT / "dataset4"
WORKING_DIR = DATASET_DIR / "working"

ENSEMBL_HEADERS = {"Accept": "application/json"}
ENSEMBL_LOOKUP_URL = "https://rest.ensembl.org/lookup/id/{gene_id}"
ENSEMBL_SEQUENCE_URL = "https://rest.ensembl.org/sequence/id/{object_id}"

def fetch_sequence_example(ensembl_id: str) -> dict:
    transcript_id = None
    try:
        lookup_response = requests.get(
            ENSEMBL_LOOKUP_URL.format(gene_id=ensembl_id),
            headers=ENSEMBL_HEADERS,
            params={"expand": 1},
            timeout=30,
        )
        lookup_response.raise_for_status()
        lookup_data = lookup_response.json()
        transcript_id = lookup_data.get("canonical_transcript")
        if not transcript_id:
            transcripts = lookup_data.get("Transcript") or []
            if transcripts:
                transcript_id = transcripts[0].get("id")
        if not transcript_id:
            raise ValueError(f"No transcript available for {ensembl_id}")
        transcript_id = transcript_id.split(".", 1)[0]

        sequence_response = requests.get(
            ENSEMBL_SEQUENCE_URL.format(object_id=transcript_id),
            headers=ENSEMBL_HEADERS,
            params={"type": "cdna"},
            timeout=30,
        )
        sequence_response.raise_for_status()
        sequence_data = sequence_response.json()
        sequence = sequence_data.get("seq")
        if sequence:
            return {
                "ensembl_id": ensembl_id,
                "transcript_id": transcript_id,
                "sequence_length": len(sequence),
                "sequence_prefix": sequence[:40],
                "sequence": sequence,
            }
    except Exception as exc:  # noqa: BLE001
        return {
            "ensembl_id": ensembl_id,
            "transcript_id": transcript_id,
            "retrieval_example_error": str(exc),
        }
    return {}

normalized = pd.read_csv(WORKING_DIR / "normalized_rawcounts.csv")[
    ["ensembl_id", "mean_nt", "mean_unmod", "mean_amide", "mean_gna"]
]
alignment_df = pd.read_csv(DATASET_DIR / "gene_alignments3.csv")
alignment_df = alignment_df.merge(normalized, on="ensembl_id", how="left")
alignment_df["target_encoding"] = alignment_df["target_rna"]

with np.errstate(divide="ignore", invalid="ignore"):
    alignment_df["log2FC_unmod"] = np.log2(alignment_df["mean_unmod"] / alignment_df["mean_nt"])
    alignment_df["log2FC_amide"] = np.log2(alignment_df["mean_amide"] / alignment_df["mean_nt"])
    alignment_df["log2FC_gna"] = np.log2(alignment_df["mean_gna"] / alignment_df["mean_nt"])

dup_counts = alignment_df["target_rna"].value_counts()
alignment_df["off_target"] = alignment_df["target_rna"].map(dup_counts).gt(1).astype(int)
alignment_df.to_csv(WORKING_DIR / "alignment_with_labels.csv", index=False)

candidate_ids = []
seen = set()
for value in alignment_df["ensembl_id"].dropna().astype(str):
    gene_id = value.strip()
    if not gene_id.startswith("ENSG") or gene_id in seen:
        continue
    seen.add(gene_id)
    candidate_ids.append(gene_id)
    if len(candidate_ids) >= 12:
        break

sequence_example = {"retrieval_example_error": "No candidate Ensembl IDs were found."}
for gene_id in candidate_ids:
    sequence_example = fetch_sequence_example(gene_id)
    if sequence_example.get("sequence"):
        break

sequence_json = dict(sequence_example)
json_path = WORKING_DIR / "sequence_example.json"
fasta_path = WORKING_DIR / "sequence_example.fasta"
if sequence_example.get("sequence"):
    fasta_path.write_text(
        f">{sequence_example['ensembl_id']}|{sequence_example.get('transcript_id', 'unknown')}\n{sequence_example['sequence']}\n",
        encoding="utf-8",
    )
    sequence_json["fasta_path"] = str(fasta_path)
elif fasta_path.exists():
    fasta_path.unlink()

json_path.write_text(json.dumps(sequence_json, indent=2), encoding="utf-8")

print({key: value for key, value in sequence_json.items() if key != "sequence"})
alignment_df[["ensembl_id", "target_rna", "log2FC_unmod", "log2FC_amide", "log2FC_gna", "off_target"]].head(3)


In [ ]:
# Dataset 4.3 - Generate the compact RDKit fingerprint blocks used in Dataset 4.
# Details:
# - Uses the same md0/features.ipynb RDKit + correction flow as Dataset 0.
# - Generates the compact 96-value fingerprint block for every comp_name in this dataset.
# - Writes working/fingerprints_output.csv, fp_corrected1.csv, fp_corrected2.csv, and raw_data_fp.csv.

from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

DATA_ROOT = Path("/media/volume/sirna-features")
SHARED_DATA_DIR = DATA_ROOT / "shared_data"
RDKIT_DIR = SHARED_DATA_DIR / "fingerprints"
published_comp_names = set(
    pd.read_csv(DATASET_DIR / "raw_data_4.csv", usecols=["comp_name"])["comp_name"]
)

def fingerprint_to_uint8(fp) -> list[int]:
    bits = list(fp)
    while len(bits) % 8:
        bits.append(0)
    return np.packbits(bits).tolist()

def calculate_fingerprint_from_pdb(pdb_file, nBits=256, radius=1):
    mol = Chem.MolFromPDBFile(str(pdb_file))
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits)
    if mol is None:
        return None
    return generator.GetFingerprint(mol)

files = ["U.pdb", "AG.pdb", "A.pdb", "C.pdb", "GNAU.pdb", "G.pdb", "R3-R4.pdb"]
fingerprints = {}
for file_name in files:
    pdb_file = RDKIT_DIR / file_name
    if file_name in ["AG.pdb", "R3-R4.pdb"]:
        fingerprints[file_name] = calculate_fingerprint_from_pdb(pdb_file, nBits=512, radius=2)
    else:
        fingerprints[file_name] = calculate_fingerprint_from_pdb(pdb_file)

output_data = []
for file_name, fp in fingerprints.items():
    uint8_vector = fingerprint_to_uint8(fp)
    output_data.append([file_name, list(uint8_vector)])
fingerprints_df = pd.DataFrame(output_data, columns=["File", "uint8_vector"])
fingerprints_df.to_csv(WORKING_DIR / "fingerprints_output.csv", index=False)

fingerprints = pd.read_csv(WORKING_DIR / "fingerprints_output.csv")
fingerprints["File"] = fingerprints["File"].str.replace(".pdb", "", regex=False)
fingerprints["uint8_vector"] = fingerprints["uint8_vector"].str.strip("[]").apply(lambda x: list(map(int, x.split(","))))
uint8_expanded = fingerprints["uint8_vector"].apply(pd.Series)
fingerprints = fingerprints.drop("uint8_vector", axis=1).join(uint8_expanded)
fingerprints.columns = ["base"] + [f"uint8_{i}" for i in range(uint8_expanded.shape[1])]
fingerprints_full_data = fingerprints.dropna().reset_index(drop=True)
fingerprints_less_data = fingerprints[fingerprints.isnull().any(axis=1)].iloc[:, :-32].reset_index(drop=True)
fingerprints_full_data.to_csv(WORKING_DIR / "fp_corrected1.csv", index=False)
fingerprints_less_data.to_csv(WORKING_DIR / "fp_corrected2.csv", index=False)

comp_records = []
for row in alignment_df.itertuples(index=False):
    for suffix in ("amide", "gna", "unmod"):
        comp_records.append({"comp_name": f"{row.ensembl_id}_{suffix}"})
df = pd.DataFrame(comp_records)

df_fp = pd.read_csv(WORKING_DIR / "fp_corrected1.csv")
output = pd.DataFrame()
for comp in df["comp_name"]:
    if "amide" in comp:
        temp = df_fp.loc[df_fp["base"] == "R3-R4"].copy()
    else:
        temp = df_fp.loc[df_fp["base"] == "AG"].copy()
    temp.loc[:, "comp_name"] = comp
    output = pd.concat([output, temp], ignore_index=True)
output = output[["comp_name"] + [col for col in output.columns if col not in {"comp_name", "base"}]]

fp_corrected = pd.read_csv(WORKING_DIR / "fp_corrected2.csv")
gna_data = fp_corrected[fp_corrected["base"] == "GNAU"].drop(columns="base").reset_index(drop=True)
u_data = fp_corrected[fp_corrected["base"] == "U"].drop(columns="base").reset_index(drop=True)
new_columns = [f"uint8_{i}" for i in range(64, 64 + gna_data.shape[1])]
gna_data.columns = new_columns
u_data.columns = new_columns

gna_rows = output["comp_name"].str.contains("gna", case=False)
output_gna = output[gna_rows].reset_index(drop=True)
output_u = output[~gna_rows].reset_index(drop=True)
gna_data = pd.concat([gna_data] * len(output_gna), ignore_index=True)
u_data = pd.concat([u_data] * len(output_u), ignore_index=True)
output_gna = pd.concat([output_gna, gna_data], axis=1)
output_u = pd.concat([output_u, u_data], axis=1)
compact_fp_df = pd.concat([output_gna, output_u]).reset_index(drop=True).sort_values(by="comp_name")
compact_fp_df = compact_fp_df[compact_fp_df["comp_name"].isin(published_comp_names)].copy()
compact_fp_df = compact_fp_df.drop_duplicates(subset="comp_name", keep="first")
compact_fp_df.to_csv(WORKING_DIR / "raw_data_fp.csv", index=False)

print(fingerprints_full_data.shape, fingerprints_less_data.shape, compact_fp_df.shape)
compact_fp_df


In [ ]:
# Dataset 4.4 - Build raw_data_4.csv from gene index bytes, compact fingerprints, and labels.
# Details:
# - Converts each Ensembl ID into the three gene_idx bytes.
# - Attaches the comp_name-specific 96-value fingerprint block generated in 4.3.
# - Writes working/raw_data_4.csv.

import pandas as pd

def ensembl_to_uint8_triplet(ensembl_id: str) -> list[int]:
    gene_number = int(ensembl_id.replace("ENSG", "").lstrip("0") or "0")
    return [(gene_number >> 16) & 0xFF, (gene_number >> 8) & 0xFF, gene_number & 0xFF]

label_records = []
for row in alignment_df.itertuples(index=False):
    for suffix in ("amide", "gna", "unmod"):
        label_records.append({
            "comp_name": f"{row.ensembl_id}_{suffix}",
            "log2FC": float(getattr(row, f"log2FC_{suffix}")),
            "off_target": int(row.off_target),
        })
published_comp_names = set(
    pd.read_csv(DATASET_DIR / "raw_data_4.csv", usecols=["comp_name"])["comp_name"]
)
label_frame = pd.DataFrame(label_records).drop_duplicates(subset="comp_name", keep="first")
label_frame = label_frame[label_frame["comp_name"].isin(published_comp_names)].copy()

raw_data_4 = label_frame.copy()
raw_data_4["ensembl_id"] = raw_data_4["comp_name"].str.split("_").str[0]
raw_data_4[["gene_idx0", "gene_idx1", "gene_idx2"]] = pd.DataFrame(
    raw_data_4["ensembl_id"].apply(ensembl_to_uint8_triplet).tolist(),
    index=raw_data_4.index,
)
raw_data_4 = raw_data_4.merge(compact_fp_df, on="comp_name", how="left")

fp_cols = [column for column in compact_fp_df.columns if column != "comp_name"]
raw_data_4 = raw_data_4[
    ["comp_name", "gene_idx0", "gene_idx1", "gene_idx2"] + fp_cols + ["log2FC", "off_target"]
]
duplicate_entries_path = DATA_ROOT / "shared_data" / "duplicate_entries.csv"
if duplicate_entries_path.exists():
    # Entries were removed with duplicate target sequences
    duplicate_entries = pd.read_csv(duplicate_entries_path)
    raw_data_4 = raw_data_4[~raw_data_4["comp_name"].isin(duplicate_entries["comp_name"])].copy()
raw_data_4 = raw_data_4.sort_values("comp_name").reset_index(drop=True)
raw_data_4.to_csv(WORKING_DIR / "raw_data_4.csv", index=False)
print(raw_data_4.shape)
raw_data_4.head(3)


In [ ]:
# Dataset 4.cleanup - Release large objects and reset notebook state.
# Details:
# - Frees common large tables from memory if they were created above.
# - Closes open matplotlib figures.
# - Resets PyMOL state when it was used in this notebook.

import gc

LARGE_NAMES = [
    "raw_counts",
    "counts_only",
    "normalized",
    "alignment_df",
    "label_frame",
    "distance_df",
    "distance_frames",
    "compact_fp_df",
    "rna_xyz_df",
    "residue_xyz_df",
    "guide_df",
    "target_snapshot_df",
    "raw_data_0",
    "raw_data_1",
    "raw_data_2",
    "raw_data_3",
    "raw_data_4",
    "raw_data_3_encoding",
    "merged_frames",
    "branch_frames",
]

for _name in LARGE_NAMES:
    if _name in globals():
        del globals()[_name]

try:
    import matplotlib.pyplot as plt
    plt.close("all")
except Exception:
    pass

try:
    from pymol import cmd
    cmd.delete("all")
    cmd.reinitialize()
except Exception:
    pass

freed = gc.collect()
print("Cleanup complete.", {"gc_objects_collected": freed})
